# Enterprise Document RAG Pipeline with ChromaDB & Citations

**Tech Stack:** Python, LangChain, ChromaDB, Google Gemini (Free Tier) / OpenAI, LCEL  

---

## 📌 Project Overview
In this project, I developed an end-to-end, production-ready **Retrieval-Augmented Generation (RAG)** pipeline designed to ingest unstructured enterprise documents (PDF & Word documents), build semantic vector indices, and generate verifiable, hallucination-free answers backed by precise source citations.

Large Language Models (LLMs) often hallucinate or lack domain-specific private context. To solve this, I designed a modular architecture leveraging **LangChain Expression Language (LCEL)**, **OpenAI Embeddings**, and a persistent **Chroma Vector Database**.

### 🏗️ Pipeline Architecture
```
[Unstructured Documents: PDF / DOCX]
         │
         ▼
[Loaders: PyPDFLoader, Docx2txtLoader]
         │
         ▼
[Hierarchical Splitters: MarkdownHeaderTextSplitter + CharacterTextSplitter]
         │
         ▼
[Embeddings: Google Gemini / OpenAI]
         │
         ▼
[Vector Store: Persistent ChromaDB] ◄─── Document CRUD Management
         │
         ▼
[Retriever: Maximal Marginal Relevance (MMR)]
         │
         ▼
[Context Stuffing Prompt + Gemini / GPT-4 + StrOutputParser]
         │
         ▼
[Grounded Output with Resource Citations]
```

---

### 🛠️ Key Implementation Highlights
1. **Document Ingestion**: Parsing multi-format enterprise files (`.pdf` and `.docx`) with full metadata tracking.
2. **Semantic Header Chunking**: Preserving course and lecture hierarchy inside chunk metadata before applying length splitting.
3. **Dense Vector Embeddings**: Generating embeddings and benchmarking semantic similarity using cosine distance.
4. **Persistent Vector Storage**: Managing document vectors on disk with ChromaDB and supporting runtime document updates.
5. **MMR Retrieval**: Avoiding redundant context chunks by balancing relevance with diversity (`fetch_k=10, k=3, λ=0.7`).
6. **Strict Grounding & Citations**: Directing GPT-4 to only answer from context and automatically cite source lectures.


## 1. Multi-Format Document Ingestion (PDF & DOCX)

In this section, I load unstructured source documents into standard LangChain `Document` objects containing both raw text and metadata:


In [5]:
%load_ext dotenv
%dotenv

import os
import copy
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

# 1. Load PDF Document
pdf_path = "Introduction_to_Data_and_Data_Science.pdf"
if os.path.exists(pdf_path):
    loader_pdf = PyPDFLoader(pdf_path)
    pages_pdf = loader_pdf.load()
    print(f"Loaded PDF: {len(pages_pdf)} pages.")
    
    # Clean up whitespace
    pages_pdf_clean = copy.deepcopy(pages_pdf)
    for doc in pages_pdf_clean:
        doc.page_content = " ".join(doc.page_content.split())
    print("Cleaned PDF Page 1 Sample:\n", pages_pdf_clean[0].page_content[:300])
else:
    print(f"Notice: {pdf_path} not found in current directory.")

print("\n" + "="*50 + "\n")

# 2. Load DOCX Document
docx_path = "Introduction_to_Data_and_Data_Science_2.docx"
loader_docx = Docx2txtLoader(docx_path)
pages_docx = loader_docx.load()
print(f"Loaded DOCX: {len(pages_docx)} document(s).")
print("DOCX Content Sample:\n", pages_docx[0].page_content[:300])


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
Loaded PDF: 6 pages.
Cleaned PDF Page 1 Sample:
 Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a dist


Loaded DOCX: 1 document(s).
DOCX Content Sample:
 # Introduction to Data and Data Science

## Analysis vs Analytics

Alright! So…
Let’s discuss the not-so-obvious differences
between the terms analysis and analytics.
Due to the similarity of the words, some people
believe they share the same meaning, and thus
use them interchangeably. Technically, 


## 2. Hierarchical Document Splitting & Metadata-Preserving Chunking

Effective chunking requires balancing semantic coherence with token limits. Here, I implement a two-stage chunking strategy:
1. `MarkdownHeaderTextSplitter`: Retains structural headings (`# Course Title`, `## Lecture Title`) directly inside chunk metadata.
2. `CharacterTextSplitter`: Splits larger body paragraphs into consistent chunks with an overlap buffer to prevent context boundary loss.


In [6]:
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_text_splitters.character import CharacterTextSplitter

# 1. Split on Markdown Headers to preserve lecture metadata
headers_to_split_on = [
    ("#", "Course Title"),
    ("##", "Lecture Title")
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
pages_md_split = md_splitter.split_text(pages_docx[0].page_content)

# Clean excessive whitespace
for doc in pages_md_split:
    doc.page_content = " ".join(doc.page_content.split())

print(f"Created {len(pages_md_split)} structural header chunks.")
print("Header Chunk Example Metadata:", pages_md_split[0].metadata)

# 2. Secondary split with CharacterTextSplitter for length consistency
char_splitter = CharacterTextSplitter(
    separator=".",
    chunk_size=500,
    chunk_overlap=50
)
chunks = char_splitter.split_documents(pages_md_split)

print(f"Total processed chunks ready for indexing: {len(chunks)}")
print("\nChunk 0 Sample:\n", chunks[0].page_content)
print("Chunk 0 Metadata:", chunks[0].metadata)


Created 2 structural header chunks.
Header Chunk Example Metadata: {'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}
Total processed chunks ready for indexing: 20

Chunk 0 Sample:
 Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis
Chunk 0 Metadata: {'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}


## 3. Dense Vector Embeddings (OpenAI)

I vectorize text chunks using OpenAI's dense embedding model (`text-embedding-ada-002`) and compute cosine similarity across sample chunks to verify semantic clustering.


In [7]:
import os
import numpy as np

# Select Free Google Gemini Embeddings or OpenAI based on configured API key
if os.getenv("GOOGLE_API_KEY"):
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    print("Initialized Google Gemini Embeddings (models/gemini-embedding-001) [Free Tier]")
else:
    from langchain_openai import OpenAIEmbeddings
    embedding = OpenAIEmbeddings(model="text-embedding-ada-002")
    print("Initialized OpenAI Embeddings (text-embedding-ada-002)")

# Embed sample chunks to verify dimensionality
vector1 = embedding.embed_query(chunks[3].page_content)
vector2 = embedding.embed_query(chunks[5].page_content)
vector3 = embedding.embed_query(chunks[18].page_content)

print(f"Vector dimensionality: {len(vector1)}")

# Compute cosine similarities (dot product of normalized vectors)
v1_norm = np.array(vector1) / np.linalg.norm(vector1)
v2_norm = np.array(vector2) / np.linalg.norm(vector2)
v3_norm = np.array(vector3) / np.linalg.norm(vector3)

dot_1_2 = np.dot(v1_norm, v2_norm)
dot_1_3 = np.dot(v1_norm, v3_norm)
dot_2_3 = np.dot(v2_norm, v3_norm)

print(f"Cosine Similarity (Chunk 3 & Chunk 5): {dot_1_2:.4f}")
print(f"Cosine Similarity (Chunk 3 & Chunk 18): {dot_1_3:.4f}")
print(f"Cosine Similarity (Chunk 5 & Chunk 18): {dot_2_3:.4f}")


Initialized Google Gemini Embeddings (models/gemini-embedding-001) [Free Tier]
Vector dimensionality: 3072
Cosine Similarity (Chunk 3 & Chunk 5): 0.8113
Cosine Similarity (Chunk 3 & Chunk 18): 0.6164
Cosine Similarity (Chunk 5 & Chunk 18): 0.6033


## 4. Persistent Vector Database Storage (ChromaDB)

I index the document chunks into a persistent on-disk Chroma collection at `./intro-to-ds-lectures` for low-latency retrieval.


In [8]:
# Modern import with fallback
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

persist_dir = "./intro-to-ds-lectures"

# Create and persist the vectorstore
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_dir
)

doc_count = len(vectorstore.get()['documents'])
print(f"Chroma vectorstore initialized and persisted at '{persist_dir}'.")
print(f"Total documents stored: {doc_count}")


Chroma vectorstore initialized and persisted at './intro-to-ds-lectures'.
Total documents stored: 20


## 5. Vector Store Lifecycle Management (CRUD)

Production vector stores must handle dynamic document lifecycles. Here, I demonstrate adding new documents, updating modified documents, and deleting obsolete entries by ID.


In [9]:
from langchain_core.documents import Document

# 1. Add a new document
new_doc = Document(
    page_content="Alright! So... Let's discuss the not-so-obvious tools used in data science.",
    metadata={
        "Course Title": "Introduction to Data and Data Science",
        "Lecture Title": "Programming Languages & Software Employed in Data Science - All the Tools You Need"
    }
)

added_ids = vectorstore.add_documents([new_doc])
test_id = added_ids[0]
print(f"Added document with ID: {test_id}")
print("Fetched newly added doc:", vectorstore.get(ids=[test_id])['documents'])

# 2. Update document
updated_doc = Document(
    page_content="Great! We hope we gave you a good idea about the applicability of frequently used data science tools.",
    metadata={
        "Course Title": "Introduction to Data and Data Science",
        "Lecture Title": "Programming Languages & Software Employed in Data Science - All the Tools You Need"
    }
)
vectorstore.update_document(document_id=test_id, document=updated_doc)
print("Fetched updated doc:", vectorstore.get(ids=[test_id])['documents'])

# 3. Delete document
vectorstore.delete(ids=[test_id])
remaining = vectorstore.get(ids=[test_id])['documents']
print(f"Document {test_id} deleted. Remaining matches: {len(remaining)}")


Added document with ID: b52c1d67-dff8-483f-b9c4-21e3e91df126
Fetched newly added doc: ["Alright! So... Let's discuss the not-so-obvious tools used in data science."]
Fetched updated doc: ['Great! We hope we gave you a good idea about the applicability of frequently used data science tools.']
Document b52c1d67-dff8-483f-b9c4-21e3e91df126 deleted. Remaining matches: 0


## 6. Retrieval Optimization: Similarity Search vs. Maximal Marginal Relevance (MMR)

Standard similarity search can return repetitive, near-identical chunks. I implement **Maximal Marginal Relevance (MMR)** to balance relevance with diversity (`fetch_k=10, k=3, lambda_mult=0.7`).


In [10]:
question = "What programming languages do data scientists use?"

# 1. Similarity Search
docs_similarity = vectorstore.similarity_search(query=question, k=3)
print("=== Similarity Search Results (Top 3) ===")
for i, d in enumerate(docs_similarity):
    print(f"[{i+1}] {d.page_content[:150]}... (Lecture: {d.metadata.get('Lecture Title')})")

print("\n" + "="*50 + "\n")

# 2. Maximal Marginal Relevance (MMR)
docs_mmr = vectorstore.max_marginal_relevance_search(
    query=question,
    k=3,
    fetch_k=10,
    lambda_mult=0.7
)
print("=== Maximal Marginal Relevance (MMR) Results (Top 3) ===")
for i, d in enumerate(docs_mmr):
    print(f"[{i+1}] {d.page_content[:150]}... (Lecture: {d.metadata.get('Lecture Title')})")


=== Similarity Search Results (Top 3) ===
[1] As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipul... (Lecture: Programming Languages & Software Employed in Data Science - All the Tools You Need)
[2] Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of dat... (Lecture: Programming Languages & Software Employed in Data Science - All the Tools You Need)
[3] What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have n... (Lecture: Programming Languages & Software Employed in Data Science - All the Tools You Need)


=== Maximal Marginal Relevance (MMR) Results (Top 3) ===
[1] As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they 

## 7. Vectorstore-Backed Retriever as an LCEL Runnable

I convert the vectorstore into a native LCEL `Retriever` runnable that can be piped directly into downstream prompts and models.


In [11]:
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'lambda_mult': 0.7}
)

# Test retriever invocation as a Runnable
retrieved_docs = retriever.invoke("What software do data scientists use?")
print(f"Retriever returned {len(retrieved_docs)} documents.")


Retriever returned 3 documents.


## 8. Prompt Engineering: Context Stuffing & Citation Enforcement

To eliminate hallucinations, I engineer a prompt template that strictly limits responses to the provided context and mandates source lecture citations.


In [20]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

TEMPLATE = """Answer the following question:
{question}

To answer the question, use only the following context:
{context}

Formatting Instructions:
- Format the response using clean bullet points with line breaks.
- List each programming language or software tool on its own bullet point.
- Provide a concise description of what each tool is used for.
- Add an empty line between points for readability.

At the end of the response, on a separate line, specify the source:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
"""

prompt_template = PromptTemplate.from_template(TEMPLATE)

# Select Free Google Gemini Chat Model or OpenAI based on configured API key
if os.getenv("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    chat = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        temperature=0,
        max_output_tokens=1024
    )
    print("Initialized Google Gemini Chat Model (gemini-3.5-flash) [Free Tier]")
else:
    from langchain_openai import ChatOpenAI
    chat = ChatOpenAI(
        model='gpt-4',
        temperature=0,
        seed=365,
        max_tokens=250
    )
    print("Initialized OpenAI Chat Model (gpt-4)")

# Helper function to format retrieved documents with metadata
def format_docs(docs):
    formatted = []
    for d in docs:
        lecture = d.metadata.get('Lecture Title', 'General')
        formatted.append(f"[Lecture: {lecture}]\n{d.page_content}")
    return "\n\n".join(formatted)


Initialized Google Gemini Chat Model (gemini-3.5-flash) [Free Tier]


## 9. End-to-End Declarative LCEL RAG Chain

Here, I compose the complete RAG pipeline into a single declarative LCEL statement (`retriever | prompt | model | parser`) and execute it against user queries.


In [22]:
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser

# Universal parser supporting both string and content block outputs
def parse_response(response):
    content = getattr(response, 'content', response)
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and 'text' in item:
                parts.append(item['text'])
            elif isinstance(item, str):
                parts.append(item)
        return "".join(parts).strip()
    return str(content).strip()

# Assemble the declarative LCEL RAG chain
rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | prompt_template
    | chat
    | parse_response
)

# Query the complete RAG system!
user_query = "What software do data scientists use?"

response_text = rag_chain.invoke(user_query)

# Display beautifully formatted markdown with bold text, clean newlines, and bullet points
display(Markdown(f"""### Grounded RAG Response

{response_text}
"""))


### Grounded RAG Response

Based on the provided context, data scientists primarily use programming languages and tools such as **R** and **Python**. These are the most popular, adaptable tools used to manipulate data, perform mathematical and statistical computations, solve business and data-related problems, and create specific, ad-hoc tools for projects. They are also integrated within multiple data and data science software platforms.

Resources: *Lecture: Programming Languages & Software Employed in Data Science - All the Tools You Need*
